# Pre-Lesson Exercise: Organize Your Workspace Space Kick-Off

Before we get started on our course, let's take some time to set up and tidy our workspace. As an agentic twist on the matter, we will guide a Large Language Model (LLM) through a series of prompt refinements to create a practical plan to organize your personal work area.

## Outline

1. Start with a generic prompt
2. Add a professional role
3. Introduce concrete constraints
4. Bonus: Apply what you've learned

## API configuration

Run this notebook with the repository's Python environment. Set `OPENAI_API_KEY` for
direct OpenAI access, or set `VOCAREUM_API_KEY` for the Udacity classroom gateway.
The notebook loads the nearest local `.env` from the working directory or its parents
and preserves variables already set in the kernel environment.

When both keys are available, direct OpenAI is the default. Set
`OPENAI_BASE_URL=https://openai.vocareum.com/v1` to select Vocareum explicitly.
Only the official OpenAI and Vocareum endpoints are supported.
A classroom key stored in `OPENAI_API_KEY` is recognized by its `voc-` prefix.
Keep credentials in the environment or ignored `.env` file.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI, OpenAIError

In [ ]:
load_dotenv(find_dotenv(usecwd=True))

openai_url = "https://api.openai.com/v1"
vocareum_url = "https://openai.vocareum.com/v1"
openai_key = (os.getenv("OPENAI_API_KEY") or "").strip()
vocareum_key = (os.getenv("VOCAREUM_API_KEY") or "").strip()
base_url = (os.getenv("OPENAI_BASE_URL") or "").strip().rstrip("/")

if base_url and base_url not in {openai_url, vocareum_url}:
    raise ValueError("OPENAI_BASE_URL must use the official OpenAI or Vocareum endpoint.")

if openai_key.startswith("voc-"):
    vocareum_key = vocareum_key or openai_key
    openai_key = ""

if base_url == vocareum_url or (not base_url and not openai_key and vocareum_key):
    api_key = vocareum_key
    base_url = vocareum_url
else:
    api_key = openai_key

if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or VOCAREUM_API_KEY for the selected provider.")

client = OpenAI(api_key=api_key, base_url=base_url or openai_url)

In [ ]:
# No changes needed in this cell
from enum import Enum


class OpenAIModels(str, Enum):
    GPT_4O_MINI = "gpt-4o-mini"
    GPT_41_MINI = "gpt-4.1-mini"
    GPT_41_NANO = "gpt-4.1-nano"


MODEL = OpenAIModels.GPT_41_NANO


def get_completion(system_prompt, user_prompt, model=MODEL):
    """
    Function to get a completion from the OpenAI API.
    Args:
        system_prompt: The system prompt
        user_prompt: The user prompt
        model: The model to use (default is gpt-4.1-nano)
    Returns:
        The completion text
    """
    messages = [
        {"role": "user", "content": user_prompt},
    ]
    if system_prompt is not None:
        messages = [
            {"role": "system", "content": system_prompt},
            *messages,
        ]
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.7,
        )
        return response.choices[0].message.content
    except OpenAIError as exc:
        raise RuntimeError(
            f"The model request failed ({type(exc).__name__}). "
            "Check your provider configuration and model access."
        ) from None


def display_responses(*args):
    """Helper function to display responses as Markdown, horizontally."""
    markdown_string = "<table><tr>"
    # Headers
    for arg in args:
        markdown_string += f"<th>System Prompt:<br />{arg['system_prompt']}<br /><br />"
        markdown_string += f"User Prompt:<br />{arg['user_prompt']}</th>"
    markdown_string += "</tr>"
    # Rows
    markdown_string += "<tr>"
    for arg in args:
        markdown_string += f"<td>Response:<br />{arg['response']}</td>"
    markdown_string += "</tr></table>"
    display(Markdown(markdown_string))

## 1. Generic Prompt

First, let's see what the model produces with a basic prompt.

In [ ]:
# No changes needed in this cell
plain_system_prompt = "You are a helpful assistant."  # A generic system prompt
user_prompt = "Give me a simple plan to declutter and organize my workspace."

print(f"Sending prompt to {MODEL} model...")
baseline_response = get_completion(plain_system_prompt, user_prompt)
print("Response received!\n")

display_responses(
    {
        "system_prompt": plain_system_prompt,
        "user_prompt": user_prompt,
        "response": baseline_response,
    }
)

## 2. Add a Professional Role

Now, let's add a professional role to see how it affects the response.

<div style="color: red">NOTE: We will use the same user prompt for these examples. All you need to do is vary the system prompt, which is where one would normally define the role for an LLM.</div>

In [ ]:
role_system_prompt = "You are a professional organizer who specializes in small workspaces."

print("Sending prompt with professional role...")
role_response = get_completion(role_system_prompt, user_prompt)
print("Response received!\n")

# Show last two prompts and responses
display_responses(
    {
        "system_prompt": plain_system_prompt,
        "user_prompt": user_prompt,
        "response": baseline_response,
    },
    {
        "system_prompt": role_system_prompt,
        "user_prompt": user_prompt,
        "response": role_response,
    },
)

## 3. Add Constraints

Let's add specific constraints to see how the model prioritizes tasks. Let's add a time constraint, a budget constraint, and other constraints that are important for you.

In [ ]:
constraints_system_prompt = (
    f"{role_system_prompt} Plan for at most 15 minutes and a $20 budget. "
    "Assume there is limited floor space and preserve sentimental items. "
    "Prioritize clearing the desk and show the time and estimated cost of each step."
)

print("Sending prompt with constraints...")
constraints_response = get_completion(constraints_system_prompt, user_prompt)
print("Response received!\n")

# Show last two prompts and responses
display_responses(
    {
        "system_prompt": role_system_prompt,
        "user_prompt": user_prompt,
        "response": role_response,
    },
    {
        "system_prompt": constraints_system_prompt,
        "user_prompt": user_prompt,
        "response": constraints_response,
    },
)

## Bonus: Apply what you've learned

Try crafting a prompt for one of your own ideas or even a different organization task (e.g., digital file cleanup, closet overhaul)!

In [ ]:
custom_system_prompt = (
    "You are a digital organizer who designs simple filing systems. "
    "Offer a timed checklist and briefly explain the priority of each step."
)
user_prompt = (
    "Help me organize the digital files in my Downloads folder in 10 minutes. "
    "Use built-in tools and keep the original files recoverable."
)

print("Sending custom prompt...")
custom_response = get_completion(custom_system_prompt, user_prompt)
print("Response received!\n")

display_responses(
    {
        "system_prompt": custom_system_prompt,
        "user_prompt": user_prompt,
        "response": custom_response,
    }
)

## Summary

In this exercise, we explored how different prompt refinements affect the output of an LLM:

1. **Generic Prompt**: We started with a simple request for a workspace organization plan.
2. **Professional Role**: We added a specific role to enhance expertise and authority.
3. **Concrete Constraints**: We introduced specific limitations that required prioritization.
4. **Transfer**: We reused the role-and-constraints pattern to organize digital files.

These techniques demonstrate how prompt engineering can significantly improve the usefulness and relevance of AI-generated content for specific needs.

Excellent Work! 🚀